Create a command‑line tool that reads in a list of software packages and their dependencies, then allows the user to:

add a package with its dependencies

list all packages

query the full dependency chain of a package

detect whether adding a new dependency would create a cycle

Example interaction (purely illustrative):
```bash
> add A B C
> add B D
> add C D E
> deps A
D, E, B, C
> checkcycle A A
cycle detected
```

In [38]:
prompt = """
Please input one of the following commands to determine package dependency: \n
* add X Y ... : Add package X with dependency of Y (or more) \n
* deps X: List the dependencies of X, regardless of level \n
* checkcycle X Y: Checks the current package dependencies to see if making X depend on Y would introduce a cycle \n
* exit: exits the program \n"""
# user_input = input(prompt)
user_input = "add A B C".split(" ") # Used for testing
graph = {}

def add(processed_input, input_graph):
    # Enact the add functionality, specifically add the nodes to the current graph in an appropriate order
    head = processed_input[1]
    children = set(processed_input[2:]) # For future, ensure there are actually children
    if head in input_graph.keys() and input_graph[head]!= None:
        input_graph[head].union(children)
        print(f"Dependencies updated for {head}")
        print(f"{head} has the following dependencies: {input_graph[head]}")
    else:
        input_graph[head] = children
        print(f"Dependencies added for {head}")
        print(f"{head} has the following dependencies: {input_graph[head]}")
    for child in children:
        input_graph[child] = None

add(user_input, graph)
user_input = "add C D".split(" ")
add(user_input)

def deps(processed_input, input_graph):
    # Utilize algorithm to print out all dependencies for a given package
    head = processed_input[1]
    queue = list(graph[head])
    to_print = ""
    while queue:
        child = queue.pop(0)
        print(child)
        to_print = to_print +" "+ child # Has a buggy extra space prepending, fix later
        if child in graph.keys() and graph[child] != None:
            queue.extend(list(graph[child]))

    print(f"{head} has the following dependencies:")
    print(to_print)

user_input = "deps A".split(" ")
deps(user_input, graph)

def checkcycle(processed_input, input_graph):
    # Check if the supposed dependency would create a cycle
    temp_graph = {key: value for key,value in graph.items()} # Make a copy of the graph

    head = processed_input[1]
    child = processed_input[2]
    add(processed_input, temp_graph)
    print(temp_graph)
    cycle_detected = False
    already_traversed = temp_graph[head]
    queue = list(temp_graph[head])
    while queue:
        queue.pop(0)
        if child in graph.keys() and graph[child] != None:
            queue.extend(list(graph[child]))
            cycle_detected = len(already_traversed.union(temp_graph[child])) < len(already_traversed) + len(temp_graph[child])
            if cycle_detected:
                print("cycle detected")
                return True
    
    print("cycle not detected")
    return False

user_input = "checkcycle C A".split(" ")
checkcycle(user_input, graph)

while user_input is not 'exit':
    processed_input = user_input.split(" ") # For future, do more regex to ensure user gives proper input
    if user_input is 'exit':
        print("Thanks for using this program.")
        break
    if processed_input[0] is 'add':
        add(processed_input, graph) # Run the add function
    elif processed_input[0] is 'deps':
        deps(processed_input, graph)
    elif processed_input[0] is 'checkcycle':
        checkcycle(processed_input, graph)

Dependencies added for A
A has the following dependencies: {'C', 'B'}


<>:74: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
<>:76: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:79: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:81: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:83: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?


TypeError: add() missing 1 required positional argument: 'input_graph'